<a href="https://colab.research.google.com/github/nroselnik/Counting-ratbones-from-owl-pukes/blob/master/Final_completed_ratbone_owl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook is a companion to a manuscript "Identifying Small Mammal Bones in Barn Owl Pellets: A Technological Approach to Ecological Data Analysis".
To run this notebook, you will need to download the model first from the github repo. This portion of the code installs all the necessary libraries needed to run the notebook. As for the connection type, please select T4 or L4 GPU (which is available on the free Colab plan).

In [ ]:
!pip install opencv-python

In [ ]:
!pip install ultralytics

In [ ]:
!pip install --upgrade ultralytics  # Upgrade to the latest version

In [ ]:
!pip install supervision

In [ ]:
import torch
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO
import supervision as sv
from collections import Counter
import os

This code helps you to upload the model that you downloaded from the Github repo. There is an alternative method, which is tosave the model in your Google Drive and mount the drive in this colab Notebook. Nothing is saved in a virtual environment and will reset if the connection is terminated or you have exceeded the runtime restriction. As a general rule of thumb, do not leave it idle for more than 30 minutes.

In [ ]:
from google.colab import files
uploaded = files.upload()


After the model is uploaded, click on the folder option at the left side. Locate the file, click the three dots and choose copy file path. Paste it at the model path in the next code.

In [ ]:
# Load the trained model
model_path = '/content/best (1).pt'  # Update with the correct path
model = YOLO(model_path)

Similar to uploading the model, you need to upload the test image into the colab VM. You can use the upload file option or re-run the previous upload command. Follow the same procedure as before to copy the file path.

In [ ]:
# Load test image
image_path = "/content/WhatsApp Image 2025-01-15 at 14.58.53_10f79d2b.jpg"  # Update with your test image
image = cv2.imread(image_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


In [ ]:
# Run inference
results = model(image, verbose=False)[0]
detections = sv.Detections.from_ultralytics(results)

In [ ]:
# Count objects per class
class_counts = Counter(detections.class_id)

# Load class names (modify according to your dataset)
class_names = model.names  # Get class names from the model

# Print summary of detected objects
print("Detected Objects Summary:")
for class_id, count in class_counts.items():
    class_name = class_names.get(class_id, f"Class {class_id}")
    print(f"{class_name}: {count} detections")

This code produces a better label and bounding boxes for the detected objects.

In [ ]:
# Annotate image with larger labels
box_annotator = sv.BoxAnnotator(thickness=6)
label_annotator = sv.LabelAnnotator(text_scale=2.0, text_thickness=3)

annotated_image = image.copy()
annotated_image = box_annotator.annotate(scene=annotated_image, detections=detections)
annotated_image = label_annotator.annotate(scene=annotated_image, detections=detections)

# Display results
plt.figure(figsize=(12, 8))
plt.imshow(annotated_image)
plt.axis("off")
plt.show()


This portion of the code is to analyze multiple pictures. For example if you have multiple sampling sessions, arrange the images into seperate folders. You then need to zip the folder to upload the test images in this environment. We have written a code so that the zip files would be automatically unzipped to test_folder.

In [ ]:
import os
os.makedirs("test folder", exist_ok=True)


In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
!unzip -q Test_images.zip -d /content/test_folder # update the file name with your own "Test_images.zip"


This follows the same method as before, only the inference is done on each images inside the folder. This section also automatically counts the total number of objects detected from the images and calculates the probability of how many individual rats that were eaten by the owls.

In [ ]:
# Load the trained model
model_path = "/content/best (1).pt"  # Update with the correct path
model = YOLO(model_path)

# Path to folder with test images
test_folder = "/content/test_folder/Test images"  # Update with the correct folder path
image_files = [f for f in os.listdir(test_folder) if f.endswith(('.jpg', '.png', '.jpeg'))]

# Set minimum confidence threshold
confidence_threshold = 0.8  # Only consider detections above 30%

# Define expected bones per rat
rat_bones = {
    "skull": 1,
    "pubis": 2,
    "femur": 2,
    "mandible": 2
}

# Initialize total bone counts across all images
total_bone_counts = Counter()

for image_file in image_files:
    image_path = os.path.join(test_folder, image_file)

    # Load image
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Run inference
    results = model(image, verbose=False)[0]

    # Extract confidence scores and filter detections
    detections = sv.Detections.from_ultralytics(results)
    mask = detections.confidence >= confidence_threshold
    filtered_detections = detections[mask]

    # Count objects per class
    class_counts = Counter(filtered_detections.class_id)
    total_bone_counts.update(class_counts)

    # Load class names (modify according to your dataset)
    class_names = model.names  # Get class names from the model

    # Print summary of detected objects
    print(f"\nDetected Objects Summary for {image_file} (Above {confidence_threshold * 100}% Confidence):")
    for class_id, count in class_counts.items():
        class_name = class_names.get(class_id, f"Class {class_id}")
        print(f"{class_name}: {count} detections")

    # Annotate image with larger labels
    box_annotator = sv.BoxAnnotator(thickness=3)
    label_annotator = sv.LabelAnnotator(text_scale=2.0, text_thickness=3)

    annotated_image = image.copy()
    annotated_image = box_annotator.annotate(scene=annotated_image, detections=filtered_detections)
    annotated_image = label_annotator.annotate(scene=annotated_image, detections=filtered_detections)

    # Display results
    plt.figure(figsize=(12, 8))
    plt.imshow(annotated_image)
    plt.axis("off")
    plt.title(f"Results for {image_file}")
    plt.show()

# Estimate rat population
skull_count = total_bone_counts.get(0, 0)  # Skull count
if skull_count > 0:
    min_estimate = skull_count
else:
    # If no skulls detected, rely only on other bones
    min_estimate = max(
        total_bone_counts.get(1, 0) // 2,  # Pubis
        total_bone_counts.get(2, 0) // 2,  # Femur
        total_bone_counts.get(3, 0) // 2   # Mandible
    )

# Calculate max estimate considering surplus bones
surpluses = {}
for bone, count in total_bone_counts.items():
    if bone != 0:  # Skulls are definitive, other bones might have surplus
        surpluses[bone] = count // rat_bones.get(class_names.get(bone, "unknown"), 2)

max_estimate = sum(surplus for surplus in surpluses.values())

# Print rat population estimate
print("\nEstimated Rat Population:")
print(f"Minimum Estimate (based on skulls): {min_estimate}")
print(f"Maximum Estimate (considering all surplus bones): {max_estimate}")
